In [7]:
!unzip -q maestro-v2.0.0-midi.zip

replace maestro-v2.0.0/maestro-v2.0.0.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [8]:
import pandas as pd, os, shutil

df = pd.read_csv("maestro-v2.0.0/maestro-v2.0.0.csv")
subset = df[df["canonical_composer"].str.contains("Chopin")].head(50)

os.makedirs("midi_songs", exist_ok=True)
for f in subset["midi_filename"]:
    shutil.copy(os.path.join("maestro-v2.0.0", f), "midi_songs/")

print("Files copied:", len(os.listdir("midi_songs")))

Files copied: 50


In [11]:
!pip install -q music21

ERROR: Operation cancelled by user


In [12]:
import music21
print(music21.__version__)

9.9.2


In [13]:
import os, glob, pickle
from music21 import converter, note, chord

print("midi_songs exists:", os.path.exists("midi_songs"))

files = sorted(glob.glob("midi_songs/*.mid*"))[:15]   # start with 15 files
print("Files found:", len(files))

notes = []
for i, file in enumerate(files):
    print(f"Parsing {i+1}/{len(files)}: {file}")
    midi = converter.parse(file)
    for el in midi.flatten().notes:
        if isinstance(el, note.Note):
            notes.append(str(el.pitch))
        elif isinstance(el, chord.Chord):
            notes.append('.'.join(str(n) for n in el.normalOrder))

print("Total notes:", len(notes))
print("Unique notes/chords:", len(set(notes)))
print("First 20:", notes[:20])

with open("notes.pkl", "wb") as f:
    pickle.dump(notes, f)

midi_songs exists: True
Files found: 15
Parsing 1/15: midi_songs/MIDI-UNPROCESSED_01-03_R1_2014_MID--AUDIO_03_R1_2014_wav--5.midi
Parsing 2/15: midi_songs/MIDI-UNPROCESSED_04-05_R1_2014_MID--AUDIO_05_R1_2014_wav--4.midi
Parsing 3/15: midi_songs/MIDI-UNPROCESSED_04-05_R1_2014_MID--AUDIO_05_R1_2014_wav--7.midi
Parsing 4/15: midi_songs/MIDI-Unprocessed_02_R3_2008_01-03_ORIG_MID--AUDIO_02_R3_2008_wav--2.midi
Parsing 5/15: midi_songs/MIDI-Unprocessed_03_R1_2009_03-08_ORIG_MID--AUDIO_03_R1_2009_03_R1_2009_04_WAV.midi
Parsing 6/15: midi_songs/MIDI-Unprocessed_046_PIANO046_MID--AUDIO-split_07-06-17_Piano-e_2-02_wav--4.midi
Parsing 7/15: midi_songs/MIDI-Unprocessed_049_PIANO049_MID--AUDIO-split_07-06-17_Piano-e_2-06_wav--5.midi
Parsing 8/15: midi_songs/MIDI-Unprocessed_04_R1_2006_01-04_ORIG_MID--AUDIO_04_R1_2006_01_Track01_wav.midi
Parsing 9/15: midi_songs/MIDI-Unprocessed_04_R2_2011_MID--AUDIO_R2-D2_03_Track03_wav.midi
Parsing 10/15: midi_songs/MIDI-Unprocessed_04_R3_2011_MID--AUDIO_R3-D2_06_T

In [14]:
import pickle, numpy as np

with open("notes.pkl", "rb") as f:
    notes = pickle.load(f)

seq_len = 100
pitchnames = sorted(set(notes))
n_vocab = len(pitchnames)

note_to_int = {n: i for i, n in enumerate(pitchnames)}
int_to_note = {i: n for n, i in note_to_int.items()}

X, y = [], []
for i in range(len(notes) - seq_len):
    X.append([note_to_int[n] for n in notes[i:i + seq_len]])
    y.append(note_to_int[notes[i + seq_len]])

X = np.array(X)
y = np.array(y)

print("n_vocab:", n_vocab)
print("X shape:", X.shape)
print("y shape:", y.shape)

with open("vocab.pkl", "wb") as f:
    pickle.dump((pitchnames, note_to_int, int_to_note, seq_len), f)
np.save("X.npy", X)
np.save("y.npy", y)

n_vocab: 1037
X shape: (43654, 100)
y shape: (43654,)


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs("/content/drive/MyDrive/music-ai", exist_ok=True)
for f in ["notes.pkl", "vocab.pkl", "X.npy", "y.npy"]:
    shutil.copy(f, "/content/drive/MyDrive/music-ai/")
print(os.listdir("/content/drive/MyDrive/music-ai"))

Mounted at /content/drive
['notes.pkl', 'vocab.pkl', 'X.npy', 'y.npy']


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
for f in ["notes.pkl", "vocab.pkl", "X.npy", "y.npy"]:
    shutil.copy("/content/drive/MyDrive/music-ai/" + f, ".")

import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import numpy as np, pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint

X = np.load("X.npy")
y = np.load("y.npy")
pitchnames, note_to_int, int_to_note, seq_len = pickle.load(open("vocab.pkl", "rb"))
n_vocab = len(pitchnames)

model = Sequential([
    Input(shape=(seq_len,)),
    Embedding(n_vocab, 100),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dropout(0.3),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(n_vocab, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
model.summary()

ckpt = ModelCheckpoint("/content/drive/MyDrive/music-ai/best_model.keras",
                       monitor="loss", save_best_only=True)

history = model.fit(X, y, epochs=30, batch_size=128, callbacks=[ckpt])

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 100)       │       103,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100, 256)       │       365,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1037)           │       266,509 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,326,881 (5.06 MB)

 Trainable params: 1,326,881 (5.06 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - loss: 5.5665
Epoch 2/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - loss: 5.2356
Epoch 3/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 5.0657
Epoch 4/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - loss: 4.9320
Epoch 5/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - loss: 4.8018
Epoch 6/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 4.6662
Epoch 7/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 4.5340
Epoch 8/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 4.4172
Epoch 9/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 4.2997
Epoch 10/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 4.1945
Epoch 11/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - loss: 4.0923
Epoch 12/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - loss: 3.9946
Epoch 13/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - loss: 3.9018
Epoch 14/30
342/342 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - loss: 3.8159
Epoch 15/30
342/342 ━━━━━━━━━

In [3]:
import numpy as np, pickle
from tensorflow.keras.models import load_model
from music21 import stream, note, chord, instrument

# load the best saved model and the vocabulary
model = load_model("/content/drive/MyDrive/music-ai/best_model.keras")
pitchnames, note_to_int, int_to_note, seq_len = pickle.load(open("vocab.pkl", "rb"))
X = np.load("X.npy")

def sample(probs, temperature=1.0):
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs) / np.sum(np.exp(probs))
    return np.random.choice(len(probs), p=probs)

# pick a random starting point from the training data
pattern = list(X[np.random.randint(0, len(X))])

generated = []
for _ in range(200):
    inp = np.array(pattern).reshape(1, seq_len)
    probs = model.predict(inp, verbose=0)[0]
    idx = sample(probs, temperature=0.8)
    generated.append(int_to_note[idx])
    pattern = pattern[1:] + [idx]

# convert to MIDI
output, offset = [], 0
for s in generated:
    if '.' in s or s.isdigit():          # chord
        ns = [note.Note(int(n) + 48) for n in s.split('.')]
        c = chord.Chord(ns)
        c.offset = offset
        output.append(c)
    else:                                # single note
        n = note.Note(s)
        n.offset = offset
        output.append(n)
    offset += 0.5

s = stream.Stream(output)
s.insert(0, instrument.Piano())
s.write("midi", fp="generated_music.mid")
print("Saved generated_music.mid")

from google.colab import files
files.download("generated_music.mid")

Saved generated_music.mid


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
!apt-get install -y fluidsynth > /dev/null 2>&1
!pip install -q pretty_midi pyfluidsynth

import pretty_midi
from IPython.display import Audio

pm = pretty_midi.PrettyMIDI("generated_music.mid")
audio = pm.fluidsynth(fs=44100)
Audio(audio, rate=44100)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.0 MB/s eta 0:00:00


In [5]:
import scipy.io.wavfile as wav, numpy as np, shutil
from google.colab import files

wav.write("generated_music.wav", 44100, (audio / np.max(np.abs(audio)) * 32767).astype(np.int16))
shutil.copy("generated_music.mid", "/content/drive/MyDrive/music-ai/")
shutil.copy("generated_music.wav", "/content/drive/MyDrive/music-ai/")
files.download("generated_music.wav")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>